<a href="https://colab.research.google.com/github/mariselvi242007/research-paper-explainer-assistant/blob/main/notebooks/research_paper_explainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Explainer Assistant

This project uses Generative AI, embeddings, vector databases, and RAG
to explain research papers and answer questions based on their content.

# Step 1: Install Libraries

In [10]:
!pip install pypdf langchain langchain-community langchain-text-splitters chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


# Step 2: Import Libraries

In [11]:
# Import required libraries

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

import chromadb

import os

# Step 3.1 — Upload the PDF

In [12]:
from google.colab import files

uploaded = files.upload()

Saving research_paper.pdf to research_paper.pdf


# Step 3.2 — Check the uploaded file

In [13]:
import os

pdf_file = list(uploaded.keys())[0]

print("PDF uploaded successfully!")
print("File name:", pdf_file)

PDF uploaded successfully!
File name: research_paper.pdf


# Step 3.3 — Read the PDF

In [14]:
reader = PdfReader(pdf_file)

print("Number of pages:", len(reader.pages))

Number of pages: 21


# Step 3.4 — Extract the text

In [15]:
text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text

print("Text extracted successfully!")
print("Number of characters:", len(text))

Text extracted successfully!
Number of characters: 55266


# Step 3.5 — See the extracted text

In [16]:
print(text[:2000])

                Journal of Basrah Researches (Sciences) 52(1), 170 (2026)  
            DOI:  https://doi.org/10.56714/bjrs.52.1.13 
*Corresponding author email: baker.1994thth@gmail.com 
 ©2022 College of Education for Pure Science, University of 
Basrah. This is an Open Access Article Under the CC by 
License the CC BY 4.0 license. 
 
  ISSN: 1817-2695 (Print); 2411-524X (Online) 
Online at: https://jou.jobrs.edu.iq 
 
Interpretable Phishing Email Detection based on 
CNN-Word2vec Models and LIME Explanations 
Bakr Thamer Hammoudi1, *, Omar Z. Akif2 
1Informatics Institute for Postgraduate Studies, University of Information Technology and Communications 
(UoITC), Baghdad, Iraq. 
2Department of Computer Science, College of Education for Pure Science (Ibn al -Haitham), University of 
Baghdad, Iraq. 
A R T I C L E  I N F O  A B S T R A C T 
Received      25 February 2026 
Revised        24 March 2026 
Accepted      24 March 2026 
Published     30 June 2026 
phishing emails are a major cy

# Step 4.1 — Create Text Splitter


In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Step 4.2 — Split the Research Paper

In [19]:
chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

Number of chunks: 70


# Step 4.3 — Display the First Chunk

In [20]:
print(chunks[0])

Journal of Basrah Researches (Sciences) 52(1), 170 (2026)  
            DOI:  https://doi.org/10.56714/bjrs.52.1.13 
*Corresponding author email: baker.1994thth@gmail.com 
 ©2022 College of Education for Pure Science, University of 
Basrah. This is an Open Access Article Under the CC by 
License the CC BY 4.0 license. 
 
  ISSN: 1817-2695 (Print); 2411-524X (Online) 
Online at: https://jou.jobrs.edu.iq 
 
Interpretable Phishing Email Detection based on 
CNN-Word2vec Models and LIME Explanations 
Bakr Thamer Hammoudi1, *, Omar Z. Akif2 
1Informatics Institute for Postgraduate Studies, University of Information Technology and Communications 
(UoITC), Baghdad, Iraq. 
2Department of Computer Science, College of Education for Pure Science (Ibn al -Haitham), University of 
Baghdad, Iraq. 
A R T I C L E  I N F O  A B S T R A C T 
Received      25 February 2026 
Revised        24 March 2026 
Accepted      24 March 2026 
Published     30 June 2026


# Step 4.4 — Display the Second Chunk

In [21]:
print(chunks[1])

Baghdad, Iraq. 
A R T I C L E  I N F O  A B S T R A C T 
Received      25 February 2026 
Revised        24 March 2026 
Accepted      24 March 2026 
Published     30 June 2026 
phishing emails are a major cybersecurity threat which 
primarily attempt to steal private information or lose 
money. Attackers counterfeit well -known organizations 
and devise sophisticated tricks to re main unnoticed, so it 
remains difficult for detection systems even with ample 
academic research. This study focuses on assessing several 
methods of vectorizing text and deep learning architecture 
in classifying emails. The main contribution of this study 
is to enhance the CNN with Word2vec embedding for 
phishing email classification. To ensure the robustness, we 
built a variety of benchmark dataset by merging eight 
public sources and promoted the feature extraction by 
some specific modifications of convolut ional filters. 
Extensive preprocessing like data augmentation to handle


# Step 4.5 — Check Chunk Sizes

In [22]:
for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i+1}: {len(chunk)} characters")

Chunk 1: 952 characters
Chunk 2: 976 characters
Chunk 3: 998 characters
Chunk 4: 993 characters
Chunk 5: 920 characters


# Step 5.1 — Load the Embedding Model

In [23]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


# Step 5.2 — Generate Embeddings for the Research Paper Chunks

In [24]:
embeddings = embedding_model.encode(chunks)

print("Embeddings generated successfully!")
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Embeddings generated successfully!
Number of embeddings: 70
Embedding dimension: 384


# Step 5.3 — Display the First Embedding

In [25]:
print(embeddings[0])

[-6.11847565e-02 -5.30764945e-02  4.67071347e-02  2.40460858e-02
  5.08149788e-02 -3.88109460e-02  5.11165522e-02 -5.20543717e-02
  7.82761276e-02 -7.11062402e-02 -1.02797765e-02  9.43890121e-03
  1.45362183e-01 -1.13730980e-02 -7.47496113e-02  1.41416965e-02
 -4.76327613e-02 -3.94511931e-02 -8.80363360e-02 -5.81803545e-02
  3.16913356e-03 -3.64719555e-02  1.23555949e-02 -5.51522598e-02
 -1.20657533e-02 -1.95967760e-02 -6.61564171e-02 -1.56174798e-03
 -9.51303393e-02 -3.91058438e-03  6.73181266e-02  6.48249015e-02
  7.48935565e-02  9.16704535e-02 -6.06395639e-02  4.64149797e-03
 -2.17062607e-02  4.03783545e-02  4.72585335e-02  6.51623681e-03
 -1.52574731e-02 -1.08590886e-01 -1.05876205e-02 -3.12548759e-03
  2.94218678e-02  4.02596816e-02  5.90588944e-03  5.42100780e-02
 -1.10737942e-01 -1.08216447e-03 -6.53164387e-02  4.50783456e-03
 -9.58993798e-04  8.10781345e-02 -7.61176944e-02 -4.88294065e-02
 -7.56690875e-02 -1.67814493e-02 -3.48182358e-02 -7.74008455e-03
  6.34405296e-03  2.01459

# Step 5.4 — Check the Embedding Shape

In [26]:
print("Embedding shape:", embeddings.shape)

Embedding shape: (70, 384)


# Step 6.1 — Import Cosine Similarity

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

print("Cosine similarity imported successfully!")

Cosine similarity imported successfully!


# Step 6.2 — Create a User Question

In [28]:
question = "What is the main objective of this research paper?"

print("Question:", question)

Question: What is the main objective of this research paper?


# Step 6.3 — Generate Embedding for the User Question

In [29]:
question_embedding = embedding_model.encode([question])

print("Question embedding generated successfully!")
print("Embedding dimension:", len(question_embedding[0]))

Question embedding generated successfully!
Embedding dimension: 384


# Step 6.4 — Calculate Cosine Similarity

In [30]:
similarity_scores = cosine_similarity(
    question_embedding,
    embeddings
)[0]

print("Similarity scores calculated successfully!")
print("Number of scores:", len(similarity_scores))

Similarity scores calculated successfully!
Number of scores: 70


# Step 6.5 — Find the Most Relevant Chunk

In [31]:
most_relevant_index = similarity_scores.argmax()

print("Most relevant chunk index:", most_relevant_index)
print("Similarity score:", similarity_scores[most_relevant_index])

Most relevant chunk index: 47
Similarity score: 0.17633386


# Step 6.6 — Display the Most Relevant Chunk

In [32]:
print(chunks[most_relevant_index])

size 
Dropout Pooling layer Results 
Acc Prec Rec F1 AUC 
Word 
Embedding 
3 3,5,7 0.6 3 MaxPooling 95.67% 92.74% 99.09% 95.81% 97.97% 
3 3,5,7 0.3 3 MaxPooling 95.71% 93.58% 98.16% 95.81% 97.81% 
3 3,5,7 0.6 3 MaxPooling 96.05% 97.74% 93.35% 95.49% 98.32% 
3 3,7,10 0.5 3 MaxPooling 97.85% 97.99% 97.20% 97.60% 99.31% Interpretable Phishing Email…                                                      J. Basrah Res. (Sci.) 52(1), 170 (2026). 
184 
 
3 3,5,7 0.7 3 MaxPooling 96.84% 97.28% 96.38% 96.83% 99.28% 
3 3,5,7 0.6 3 MaxPooling 98.72% 98.92% 98.91% 98.91% 99.90% 
Table 6 indicates that the word embedding technique attained a highest accuracy of 98.72% in 
phishing email detection. The results of the FastText method are presented in Table 7: 
Table 7. Illustrates the experiments of fastText 
Appro
ach Number 
of layers 
Filters size Dropout Pooling layer Results 
Acc Prec Rec F1 AUC 
FastText 6 2,3,4,5,7,9 0.5 6 MaxPooling 98.93% 98.99% 99.20% 99.09% 99.92%


# Step 7.1 — Create ChromaDB Client

In [33]:
chroma_client = chromadb.Client()

print("ChromaDB client created successfully!")

ChromaDB client created successfully!


# Step 7.2 — Create ChromaDB Collection

In [34]:
collection = chroma_client.get_or_create_collection(
    name="research_paper"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


# Step 7.3 — Create Unique IDs for the Chunks

In [35]:
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

print("Number of chunk IDs:", len(chunk_ids))
print("First ID:", chunk_ids[0])

Number of chunk IDs: 70
First ID: chunk_0


# Step 7.4 — Store Chunks and Embeddings in ChromaDB

In [36]:
collection.add(
    ids=chunk_ids,
    documents=chunks,
    embeddings=embeddings.tolist()
)

print("Research paper chunks stored successfully!")

Research paper chunks stored successfully!


# Step 7.5 — Verify the ChromaDB Collection

In [37]:
print("Documents stored in ChromaDB:", collection.count())

Documents stored in ChromaDB: 70


# Step 7.6 — View Stored Research Paper Data

In [38]:
stored_data = collection.get()

print("Number of documents:", len(stored_data["documents"]))
print("First document:")
print(stored_data["documents"][0])

Number of documents: 70
First document:
Journal of Basrah Researches (Sciences) 52(1), 170 (2026)  
            DOI:  https://doi.org/10.56714/bjrs.52.1.13 
*Corresponding author email: baker.1994thth@gmail.com 
 ©2022 College of Education for Pure Science, University of 
Basrah. This is an Open Access Article Under the CC by 
License the CC BY 4.0 license. 
 
  ISSN: 1817-2695 (Print); 2411-524X (Online) 
Online at: https://jou.jobrs.edu.iq 
 
Interpretable Phishing Email Detection based on 
CNN-Word2vec Models and LIME Explanations 
Bakr Thamer Hammoudi1, *, Omar Z. Akif2 
1Informatics Institute for Postgraduate Studies, University of Information Technology and Communications 
(UoITC), Baghdad, Iraq. 
2Department of Computer Science, College of Education for Pure Science (Ibn al -Haitham), University of 
Baghdad, Iraq. 
A R T I C L E  I N F O  A B S T R A C T 
Received      25 February 2026 
Revised        24 March 2026 
Accepted      24 March 2026 
Published     30 June 2026


# Step 8.1 — Create a User Question

In [39]:
question = "What is the main objective of this research paper?"

print("Question:", question)

Question: What is the main objective of this research paper?


# Step 8.2 — Generate Embedding for the Question

In [40]:
question_embedding = embedding_model.encode([question])

print("Question embedding generated successfully!")

Question embedding generated successfully!


# Step 8.3 — Search the ChromaDB Vector Database

In [41]:
results = collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=3
)

print("Relevant chunks retrieved successfully!")

Relevant chunks retrieved successfully!


# Step 8.4 — Display Retrieved Chunks

In [43]:
retrieved_chunks = results["documents"][0]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(chunk)


--- Retrieved Chunk 1 ---
size 
Dropout Pooling layer Results 
Acc Prec Rec F1 AUC 
Word 
Embedding 
3 3,5,7 0.6 3 MaxPooling 95.67% 92.74% 99.09% 95.81% 97.97% 
3 3,5,7 0.3 3 MaxPooling 95.71% 93.58% 98.16% 95.81% 97.81% 
3 3,5,7 0.6 3 MaxPooling 96.05% 97.74% 93.35% 95.49% 98.32% 
3 3,7,10 0.5 3 MaxPooling 97.85% 97.99% 97.20% 97.60% 99.31% Interpretable Phishing Email…                                                      J. Basrah Res. (Sci.) 52(1), 170 (2026). 
184 
 
3 3,5,7 0.7 3 MaxPooling 96.84% 97.28% 96.38% 96.83% 99.28% 
3 3,5,7 0.6 3 MaxPooling 98.72% 98.92% 98.91% 98.91% 99.90% 
Table 6 indicates that the word embedding technique attained a highest accuracy of 98.72% in 
phishing email detection. The results of the FastText method are presented in Table 7: 
Table 7. Illustrates the experiments of fastText 
Appro
ach Number 
of layers 
Filters size Dropout Pooling layer Results 
Acc Prec Rec F1 AUC 
FastText 6 2,3,4,5,7,9 0.5 6 MaxPooling 98.93% 98.99% 99.20% 99.09% 99.92%

# Step 8.5 — Display Search Results and Distances

In [44]:
distances = results["distances"][0]

for i, distance in enumerate(distances):
    print(f"Chunk {i+1} distance: {distance:.4f}")

Chunk 1 distance: 1.6473
Chunk 2 distance: 1.6510
Chunk 3 distance: 1.6517


# Step 9.1 — Install Gemini Library

In [45]:
!pip install -q google-genai

# Step 9.2 — Import Gemini Library

In [46]:
from google import genai

print("Gemini library imported successfully!")

Gemini library imported successfully!


# Step 9.3 — Configure Gemini API

In [73]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded successfully!")

API key loaded successfully!


# Step 9.4 — Prepare Retrieved Context

In [48]:
context = "\n\n".join(retrieved_chunks)

print("Context prepared successfully!")
print("Context length:", len(context))

Context prepared successfully!
Context length: 2909


# Step 9.5 — Create the RAG Prompt

In [49]:
prompt = f"""
You are a Research Paper Explainer Assistant.

Answer the user's question using only the information
provided in the research paper context below.

Explain the answer in simple and clear language.

If the answer is not available in the context,
say: "The information is not available in the research paper."

Research Paper Context:
{context}

User Question:
{question}
"""

print(prompt)


You are a Research Paper Explainer Assistant.

Answer the user's question using only the information
provided in the research paper context below.

Explain the answer in simple and clear language.

If the answer is not available in the context,
say: "The information is not available in the research paper."

Research Paper Context:
size 
Dropout Pooling layer Results 
Acc Prec Rec F1 AUC 
Word 
Embedding 
3 3,5,7 0.6 3 MaxPooling 95.67% 92.74% 99.09% 95.81% 97.97% 
3 3,5,7 0.3 3 MaxPooling 95.71% 93.58% 98.16% 95.81% 97.81% 
3 3,5,7 0.6 3 MaxPooling 96.05% 97.74% 93.35% 95.49% 98.32% 
3 3,7,10 0.5 3 MaxPooling 97.85% 97.99% 97.20% 97.60% 99.31% Interpretable Phishing Email…                                                      J. Basrah Res. (Sci.) 52(1), 170 (2026). 
184 
 
3 3,5,7 0.7 3 MaxPooling 96.84% 97.28% 96.38% 96.83% 99.28% 
3 3,5,7 0.6 3 MaxPooling 98.72% 98.92% 98.91% 98.91% 99.90% 
Table 6 indicates that the word embedding technique attained a highest accuracy of 98.72% in 

# Step 9.6 — Generate Answer Using Gemini

In [50]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt
)

answer = response.text

print("Answer:")
print(answer)

Answer:
Based on the research paper context provided, the main objective is to detect **phishing emails**.

To achieve this, the research focuses on:
*   **Analyzing patterns:** It examines different types of information, such as "Temporal" data (the timing and frequency of emails) and "Behavioral" data (content details like keywords, URLs, and word counts) to identify unusual behaviors.
*   **Using advanced techniques:** It tests methods like "Word Embedding" and "FastText" to see how accurately they can identify phishing attempts.
*   **Data Processing:** It outlines a specific 9-stage preprocessing pipeline to prepare and clean the data for analysis.

Ultimately, the study aims to achieve high accuracy in distinguishing phishing emails from legitimate ones.


# Step 9.7 — Create the RAG Question Answering Function

In [51]:
def ask_question(question):

    question_embedding = embedding_model.encode([question])

    results = collection.query(
        query_embeddings=question_embedding.tolist(),
        n_results=5
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a Research Paper Explainer Assistant.

Use only the research paper context to answer the question.

Explain the answer simply and clearly.

If the answer is not available in the context, say:
"The information is not available in the research paper."

Research Paper Context:
{context}

Question:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text

# Step 9.8 — Test the Research Paper Explainer

In [52]:
question = "What is the main objective of this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is the main objective of this research paper?

Answer:
Based on the research paper context provided, the main objective is to develop an efficient and **interpretable framework for detecting phishing emails**.

Specifically, the research aims to:
*   **Improve Detection Accuracy:** The paper tests different techniques like Word Embedding and FastText using Convolutional Neural Networks (CNN) to identify phishing patterns, achieving accuracy rates as high as 98.72% to 100%.
*   **Provide Interpretability:** By integrating a tool called **LIME** (Local Interpretable Model-agnostic Explanations), the researchers aim to explain *why* the model classifies an email as phishing or legitimate by assigning importance weights to specific words.
*   **Analyze Diverse Features:** The study looks at various indicators, including temporal behaviors (timing of emails), behavioral patterns (word counts and keywords), and digital signatures to distinguish between malicious and benign mes

# Step 10.1 — Ask About the Abstract

In [53]:
question = "What is the research paper about?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is the research paper about?

Answer:
Based on the research paper provided, the study focuses on the following:

*   **Phishing Email Detection:** The research aims to identify phishing attacks, which are deceptive emails used by cybercriminals to steal sensitive information.
*   **Machine Learning Techniques:** It explores different methods to analyze email text, including **Word2Vec**, **FastText**, **TF-IDF**, and **Convolutional Neural Networks (CNN)** to classify emails as either phishing or legitimate.
*   **High Accuracy:** The study achieved high success rates, with the FastText method reaching 98.93% accuracy and an external test achieving up to 100% accuracy.
*   **Model Explainability:** The researchers used a tool called **LIME** (Local Interpretable Model-agnostic Explanations) to explain *why* the model labeled an email as phishing. It does this by assigning importance to specific words (e.g., words like "fraud" or "action" might indicate a phishing attempt

# Step 10.2 — Ask About the Dataset

In [54]:
question = "What dataset was used in this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What dataset was used in this research paper?

Answer:
Based on the research paper, the researchers used an integrated dataset created by merging **eight distinct data sources**. 

The finalized dataset contains **218,956 records** (117,414 phishing and 101,542 legitimate messages). The eight specific datasets used are:

1.  **CEAS-08:** A dataset used for spam filtering (39,154 messages).
2.  **Nazario:** A collection of fraudulent emails compiled by Jose Nazario (1,565 emails).
3.  **Nigerian-Fraud:** Messages related to "Nigerian scams" or "419 scams" (3,332 messages).
4.  **SpamAssassin:** An official dataset used for training open-source spam filters (5,809 emails).
5.  **Trec05:** Data from the 2005 Text Retrieval Conference spam competition (55,990 messages).
6.  **Trec06:** Data from the 2006 TREC Spam Track competition (16,457 messages).
7.  **Trec07:** The final spam track set from the TREC conference (53,757 messages).
8.  **PhishTank:** A collaborative source of d

# Step 10.3 — Ask About the Results

In [65]:
question = "What are the main results of this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What are the main results of this research paper?

Answer:
Based on the research paper provided, the main results regarding phishing email detection are as follows:

*   **Word2vec Technique:** This method achieved a highest accuracy of **99.02%**. This result was obtained using 10 layers and a "Dual pooling" (Average + Max) layer.
*   **FastText Technique:** This method reached an accuracy of **98.93%** using 6 layers and a MaxPooling layer.
*   **Word Embedding Technique:** This technique attained a highest accuracy of **98.72%**.
*   **Comparisons to Other Methods:** The paper also references other approaches, such as "RF with FastText-CBOW," which achieved accuracies ranging from **99.18% to 99.50%** across different datasets, and an SVM method that reached **99.10%** accuracy.

In summary, the research demonstrates that using deep learning configurations (like Word2vec and FastText) can identify phishing emails with high precision, specifically achieving accuracy rates b

# Step 10.4 — Ask About Limitations

In [62]:
question = "What are the limitations of this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What are the limitations of this research paper?

Answer:
The information is not available in the research paper.


# Step 10.5 — Ask About the Conclusion

In [70]:
question = "What is the conclusion of this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 17h31m36.232077939s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3-flash', 'location': 'global'

# Step 11.1 — Test Multiple Questions

In [ ]:
questions = [
    "What is the main objective of this research paper?",
    "What methodology is used in this research paper?",
    "What dataset was used?",
    "What are the main results?",
    "What are the limitations?",
    "What is the conclusion?"
]

print("Number of questions:", len(questions))

Number of questions: 6


# Step 11.2 — Generate Answers for All Questions

In [ ]:
for question in questions:
    answer = ask_question(question)

    print("\n" + "=" * 60)
    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)


Question:
What is the main objective of this research paper?

Answer:
Based on the research paper context provided, the main objective is to **detect phishing emails** using advanced machine learning techniques.

The research aims to achieve this by:
*   **Using Word Embedding and FastText:** The study tests different models to find the most accurate way to identify phishing, reaching accuracy levels as high as 98.93%.
*   **Analyzing Temporal Behaviors:** It examines the timing and frequency of emails (such as sending time or unusual days) to discover attack patterns.
*   **Analyzing Behavioral Indicators:** It looks at digital indicators and keywords within the email content (such as subject length, urgent keywords, and URL counts) to detect suspicious patterns.
*   **Ensuring Interpretability:** As indicated by the title, the research focuses on making the detection of phishing emails "interpretable," meaning the methods used to identify the threats are clear and understandable.

Q

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [10]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

In [1]:
import chromadb

client = chromadb.Client()

print("ChromaDB connected successfully!")

ChromaDB connected successfully!


In [2]:
collection = client.get_or_create_collection(
    name="research_paper"
)

print("Collection created successfully!")

Collection created successfully!


In [3]:
print("Number of documents:", collection.count())

Number of documents: 0


In [5]:
print("embedding_model:", "embedding_model" in globals())
print("collection:", "collection" in globals())

# Check common possible names for your paper chunks
for name in ["chunks", "text_chunks", "documents", "chunks_text"]:
    if name in globals():
        print(name, "FOUND →", len(globals()[name]))

embedding_model: False
collection: True


In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [7]:
print([name for name in globals() if "chunk" in name.lower()])

[]


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [9]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [72]:
question = "What methodology is used in this research paper?"

answer = ask_question(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 17h30m58.357398146s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3-flash'